# Local Meeting Summarizer — AMD/ROCm + Local LLM

This notebook implements a **local-first** meeting transcription, speaker diarization, summarization, and document-export pipeline.

### Privacy model

The application is intended to run with **audio, transcripts, models, and results kept on the local machine**. Hugging Face is used during the **model preparation/download phase**; runtime inference can then be performed without internet access.

This notebook does **not** require network access during inference when all required model files have already been downloaded/cached.

### Main improvements

- Sequential ASR/diarization/LLM stages to keep VRAM usage controlled.
- `faster-whisper` support, with an AMD/ROCm-friendly `openai-whisper` fallback.
- Token-aware chronological transcript chunking.
- Actual optional 4-bit LLM quantization.
- Recursive Map → Reduce summarization for very long meetings.
- Strict factual rules for financial figures, decisions, owners, and deadlines.
- Speaker labels and timestamps retained throughout summarization.
- DOCX as the primary export; safer PDF Markdown escaping.
- Startup diagnostics for PyTorch, GPU, VRAM, and offline/cache state.


In [1]:
# Optional one-time model preparation:
# Run this phase while internet access is available, then use the cached/local
# models during inference.
#
# Example:
#   huggingface-cli download Qwen/Qwen2.5-7B-Instruct --local-dir ./models/qwen2.5-7b
#   huggingface-cli download pyannote/speaker-diarization-3.1 --local-dir ./models/pyannote
#
# IMPORTANT:
# pyannote/speaker-diarization-3.1 is gated. Accept its model conditions on
# Hugging Face before downloading it and cache every required dependency.
#
# After the model preparation phase, set LOCAL_MODEL_DIRS below to the local
# directories or use the normal Hugging Face cache.

import os
import gc
import re
import json
import math
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Runtime privacy setting: no Hub access is needed during inference.
# This does NOT mean models must never have been downloaded from Hugging Face.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
PYANNOTE_PIPELINE_ID = "pyannote/speaker-diarization-3.1"

# Set these to local directories if you want an explicit self-contained model
# folder. Leave as None to use the local Hugging Face cache.
LOCAL_LLM_DIR = None
LOCAL_PYANNOTE_DIR = None

# ASR:
# "auto"       -> use faster-whisper when possible; on AMD/ROCm, fall back to
#                openai-whisper if faster-whisper CUDA is unavailable.
# "faster-whisper" -> force faster-whisper.
# "openai-whisper" -> use PyTorch Whisper, which can use ROCm through torch.
ASR_BACKEND = "auto"
WHISPER_MODEL_SIZE = "large-v3"

# faster-whisper/CTranslate2 CUDA is primarily aimed at NVIDIA CUDA.
# Therefore "auto" will not falsely claim that ROCm is supported.
FASTER_WHISPER_DEVICE = "auto"       # "auto", "cuda", or "cpu"
FASTER_WHISPER_COMPUTE_TYPE = "float16"

# For openai-whisper on ROCm, float16 is normally appropriate on a supported GPU.
OPENAI_WHISPER_FP16 = True

# LLM quantization:
# "4bit" is recommended for a 20 GB GPU.
# "8bit" is a compromise.
# "none" uses FP16/BF16 and may be too large depending on runtime overhead.
LLM_QUANTIZATION = "4bit"

# Token-aware chunking.
MAP_MAX_INPUT_TOKENS = 5000
MAP_OVERLAP_TOKENS = 300
MAP_MAX_NEW_TOKENS = 700

# Recursive reduction.
REDUCE_GROUP_MAX_TOKENS = 5000
REDUCE_MAX_NEW_TOKENS = 1200

OUTPUT_DOC_TYPE = "docx"  # "docx" or "pdf"

AUDIO_FILE_PATH = "meeting.mp3"

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def gpu_info():
    if not torch.cuda.is_available():
        return "No PyTorch GPU detected."
    try:
        props = torch.cuda.get_device_properties(0)
        total_gb = props.total_memory / (1024**3)
        allocated_gb = torch.cuda.memory_allocated(0) / (1024**3)
        reserved_gb = torch.cuda.memory_reserved(0) / (1024**3)
        return (
            f"{props.name} | VRAM {total_gb:.1f} GB | "
            f"allocated {allocated_gb:.2f} GB | reserved {reserved_gb:.2f} GB"
        )
    except Exception as exc:
        return f"GPU detected, but details unavailable: {exc}"

print("=== Environment ===")
print("PyTorch:", torch.__version__)
print("PyTorch GPU available:", torch.cuda.is_available())
print("GPU:", gpu_info())
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE"))
print("LLM quantization:", LLM_QUANTIZATION)
print("ASR backend:", ASR_BACKEND)


=== Environment ===
PyTorch: 2.9.1+rocm7.2.1
PyTorch GPU available: True
GPU: AMD Radeon RX 7900 XT | VRAM 20.0 GB | allocated 0.00 GB | reserved 0.00 GB
HF_HUB_OFFLINE: 1
TRANSFORMERS_OFFLINE: 1
LLM quantization: 4bit
ASR backend: auto


In [3]:
def resolve_model_path(model_id: str, local_dir: Optional[str]) -> str:
    """Use an explicit local directory when configured; otherwise use HF cache."""
    return local_dir if local_dir else model_id


def diarize_audio(audio_path: str):
    """Stage 1: Detect speaker intervals using PyAnnote."""
    from pyannote.audio import Pipeline

    print(f"\n[1/5] Loading PyAnnote diarization pipeline...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipeline_id = resolve_model_path(PYANNOTE_PIPELINE_ID, LOCAL_PYANNOTE_DIR)

    pipeline = Pipeline.from_pretrained(pipeline_id)
    if pipeline is None:
        raise RuntimeError(
            "Could not load the diarization pipeline. Make sure the model and "
            "all gated dependencies have been downloaded/cached locally."
        )

    pipeline.to(device)

    print(f"[1/5] Diarizing: {audio_path}")
    diarization_result = pipeline(audio_path)

    # Copy turns into a simple local structure before freeing the model.
    turns = [
        {
            "start": float(turn.start),
            "end": float(turn.end),
            "speaker": str(speaker),
        }
        for turn, _, speaker in diarization_result.itertracks(yield_label=True)
    ]

    del pipeline
    free_vram()
    print(f"[1/5] Detected {len(turns)} speaker turns.")
    return turns


In [4]:
def _faster_whisper_available_on_cuda() -> bool:
    """Test whether faster-whisper can actually initialize CUDA.

    On AMD/ROCm, PyTorch may expose the GPU through torch.cuda while
    CTranslate2 does not provide a compatible ROCm backend. This test prevents
    the code from pretending that ROCm == CUDA.
    """
    try:
        from faster_whisper import WhisperModel
        test_device = "cuda" if FASTER_WHISPER_DEVICE == "auto" else FASTER_WHISPER_DEVICE
        if test_device != "cuda":
            return True
        # Do not instantiate the large model here. CUDA availability alone is
        # not enough to establish CTranslate2/ROCm compatibility.
        return os.environ.get("FASTER_WHISPER_FORCE_CUDA", "0") == "1"
    except Exception:
        return False


def transcribe_with_faster_whisper(audio_path: str) -> List[Dict]:
    from faster_whisper import WhisperModel

    device = FASTER_WHISPER_DEVICE
    if device == "auto":
        device = "cuda" if _faster_whisper_available_on_cuda() else "cpu"

    compute_type = FASTER_WHISPER_COMPUTE_TYPE if device == "cuda" else "int8"

    print(f"[2/5] faster-whisper device={device}, compute_type={compute_type}")
    model = WhisperModel(
        WHISPER_MODEL_SIZE,
        device=device,
        compute_type=compute_type,
    )

    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        vad_filter=True,
        word_timestamps=False,
        condition_on_previous_text=True,
    )

    result = []
    for segment in segments:
        result.append({
            "start": float(segment.start),
            "end": float(segment.end),
            "text": segment.text.strip(),
        })

    del model
    free_vram()
    return result


def transcribe_with_openai_whisper(audio_path: str) -> List[Dict]:
    """PyTorch Whisper path; useful when running Whisper directly on ROCm."""
    import whisper

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[2/5] openai-whisper device={device}")

    model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
    result = model.transcribe(
        audio_path,
        verbose=False,
        fp16=(OPENAI_WHISPER_FP16 and device == "cuda"),
        condition_on_previous_text=True,
    )

    segments = [
        {
            "start": float(s["start"]),
            "end": float(s["end"]),
            "text": s["text"].strip(),
        }
        for s in result["segments"]
        if s["text"].strip()
    ]

    del model
    free_vram()
    return segments


def choose_asr_backend() -> str:
    if ASR_BACKEND in {"faster-whisper", "openai-whisper"}:
        return ASR_BACKEND

    # On AMD/ROCm, use PyTorch Whisper by default unless the user has explicitly
    # verified a working CTranslate2 CUDA-compatible environment.
    if torch.cuda.is_available():
        if _faster_whisper_available_on_cuda():
            return "faster-whisper"
        return "openai-whisper"

    return "faster-whisper"


def transcribe_audio(audio_path: str) -> List[Dict]:
    backend = choose_asr_backend()
    print(f"[2/5] Selected ASR backend: {backend}")

    if backend == "faster-whisper":
        return transcribe_with_faster_whisper(audio_path)

    return transcribe_with_openai_whisper(audio_path)


In [5]:
def align_speakers_to_transcript(
    whisper_segments: List[Dict],
    diarization_turns: List[Dict],
) -> str:
    """Assign the speaker with the greatest temporal overlap.

    The transcript remains chronological and every line retains timestamps.
    """
    speaker_transcript = []

    # Both lists are chronological. A moving pointer avoids O(N*M) scanning.
    j = 0

    for seg in whisper_segments:
        seg_start = seg["start"]
        seg_end = seg["end"]
        text = seg["text"].strip()
        if not text:
            continue

        while j < len(diarization_turns) and diarization_turns[j]["end"] <= seg_start:
            j += 1

        best_speaker = "UNKNOWN"
        max_overlap = 0.0

        k = j
        while k < len(diarization_turns):
            turn = diarization_turns[k]
            if turn["start"] >= seg_end:
                break

            overlap = max(
                0.0,
                min(seg_end, turn["end"]) - max(seg_start, turn["start"])
            )

            if overlap > max_overlap:
                max_overlap = overlap
                best_speaker = turn["speaker"]

            k += 1

        speaker_transcript.append(
            f"[{seg_start:08.2f}-{seg_end:08.2f}] "
            f"[{best_speaker}] {text}"
        )

    return "\n".join(speaker_transcript)


def transcribe_and_align(audio_path: str, diarization_turns: List[Dict]) -> str:
    print("\n[2/5] Transcribing audio...")
    whisper_segments = transcribe_audio(audio_path)

    print("[2/5] Aligning speakers to transcript...")
    transcript = align_speakers_to_transcript(
        whisper_segments,
        diarization_turns,
    )

    print(f"[2/5] Transcript contains {len(transcript.splitlines())} timestamped lines.")
    return transcript


In [6]:
def map_speakers_interactively(transcript: str) -> str:
    """Stage 3: Optional interactive rename of generic speaker IDs."""
    unique_speakers = sorted(set(re.findall(r"\[(SPEAKER_\d{2,}|UNKNOWN)\]", transcript)))

    if not unique_speakers:
        print("[3/5] No generic speaker tags found to map.")
        return transcript

    print("\n[3/5] Interactive Speaker Mapping")
    print("-" * 50)
    print("Enter names only if you know them. Leave blank to keep the speaker ID.")

    mapping_dict = {}
    for speaker in unique_speakers:
        real_name = input(f" -> Rename [{speaker}]: ").strip()
        if real_name:
            mapping_dict[f"[{speaker}]"] = f"[{real_name}]"

    mapped_transcript = transcript
    for old_tag, new_tag in mapping_dict.items():
        mapped_transcript = mapped_transcript.replace(old_tag, new_tag)

    print("[3/5] Speaker mapping complete.")
    return mapped_transcript


In [7]:
# ---------------------------------------------------------------------------
# Token-aware chronological chunking
# ---------------------------------------------------------------------------

def token_count(tokenizer, text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_transcript_by_tokens(
    transcript: str,
    tokenizer,
    max_tokens: int = MAP_MAX_INPUT_TOKENS,
    overlap_tokens: int = MAP_OVERLAP_TOKENS,
) -> List[str]:
    """Chunk complete transcript lines while preserving chronology.

    We never randomly shuffle transcript lines. Overlap is created by retaining
    the final few lines from the previous chunk.
    """
    lines = [line for line in transcript.splitlines() if line.strip()]
    chunks = []
    current = []
    current_tokens = 0

    for line in lines:
        line_tokens = token_count(tokenizer, line)

        # Very long single lines are split by tokenizer IDs as a last resort.
        if line_tokens > max_tokens:
            if current:
                chunks.append("\n".join(current))
                current, current_tokens = [], 0

            ids = tokenizer.encode(line, add_special_tokens=False)
            for start in range(0, len(ids), max_tokens):
                part = tokenizer.decode(ids[start:start + max_tokens])
                chunks.append(part)
            continue

        if current and current_tokens + line_tokens > max_tokens:
            chunks.append("\n".join(current))

            # Chronological overlap: retain tail lines up to overlap_tokens.
            overlap = []
            overlap_count = 0
            for old_line in reversed(current):
                n = token_count(tokenizer, old_line)
                if overlap_count + n > overlap_tokens:
                    break
                overlap.insert(0, old_line)
                overlap_count += n

            current = overlap
            current_tokens = overlap_count

        current.append(line)
        current_tokens += line_tokens

    if current:
        chunks.append("\n".join(current))

    return chunks


SYSTEM_PROMPT = """You are a meticulous executive meeting analyst.

You must summarize ONLY what is supported by the supplied transcript.

Factuality rules:
- Never invent facts, numbers, currencies, percentages, dates, deadlines, names,
  decisions, or action-item owners.
- Preserve financial figures exactly as stated where possible.
- If a value is unclear, say "Unclear in transcript".
- If an action owner is not explicitly assigned, use "Unassigned".
- Distinguish proposals from decisions.
- Do not turn speculation into fact.
- Keep the chronological context of the transcript in mind.
- Treat timestamps and speaker labels as evidence, not decoration.

Return concise Markdown using exactly these sections:
## Executive Summary
## Key Takeaways
## Decisions Made
## Financial / Business Highlights
## Risks / Issues
## Action Items
## Open Questions

For Action Items use:
- **Owner:** ...
  **Action:** ...
  **Deadline:** ...
  **Evidence:** [timestamp/speaker reference]

If no evidence exists for a section, write "None identified in this transcript segment."
"""


def build_chat_prompt(tokenizer, system_text: str, user_text: str) -> str:
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def load_local_llm():
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_id = resolve_model_path(LLM_MODEL_ID, LOCAL_LLM_DIR)
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        local_files_only=True,
    )

    quantization_config = None
    model_kwargs = {
        "device_map": "auto",
        "local_files_only": True,
    }

    if LLM_QUANTIZATION == "4bit":
        try:
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            model_kwargs["quantization_config"] = quantization_config
        except Exception as exc:
            raise RuntimeError(
                "4-bit loading was requested but BitsAndBytesConfig could not "
                f"be initialized: {exc}\n"
                "Install a bitsandbytes build compatible with your ROCm/PyTorch "
                "environment, or set LLM_QUANTIZATION='none'."
            ) from exc

    elif LLM_QUANTIZATION == "8bit":
        try:
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(load_in_8bit=True)
            model_kwargs["quantization_config"] = quantization_config
        except Exception as exc:
            raise RuntimeError(
                f"8-bit loading failed: {exc}"
            ) from exc

    elif LLM_QUANTIZATION == "none":
        model_kwargs["torch_dtype"] = (
            torch.bfloat16 if torch.cuda.is_available() else torch.float32
        )
    else:
        raise ValueError("LLM_QUANTIZATION must be '4bit', '8bit', or 'none'.")

    print(f"[4/5] Loading LLM: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

    return tokenizer, model


def generate_text(model, tokenizer, prompt: str, max_new_tokens: int) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
    )

    # device_map='auto' handles the model placement. Inputs belong on the
    # model's first input device.
    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    del inputs, output_ids
    return text


def summarize_chunk(model, tokenizer, chunk: str, index: int, total: int) -> str:
    print(f"      MAP {index}/{total}")
    prompt = build_chat_prompt(
        tokenizer,
        SYSTEM_PROMPT,
        f"""Analyze this chronological transcript segment.

Transcript segment:
{chunk}

Produce notes using the required sections. Do not summarize information that
is absent from this segment.""",
    )
    return generate_text(model, tokenizer, prompt, MAP_MAX_NEW_TOKENS)


def recursive_reduce(
    model,
    tokenizer,
    summaries: List[str],
) -> str:
    """Reduce summaries recursively so no single reduce prompt becomes huge."""
    level = 1
    current = summaries

    while len(current) > 1:
        print(f"[4/5] REDUCE level {level}: {len(current)} item(s)")
        groups = []
        current_group = []
        current_tokens = 0

        for item in current:
            n = token_count(tokenizer, item)
            if current_group and current_tokens + n > REDUCE_GROUP_MAX_TOKENS:
                groups.append(current_group)
                current_group = []
                current_tokens = 0

            current_group.append(item)
            current_tokens += n

        if current_group:
            groups.append(current_group)

        next_level = []
        for i, group in enumerate(groups, 1):
            combined = "\n\n".join(
                f"PART {j+1}:\n{txt}" for j, txt in enumerate(group)
            )

            prompt = build_chat_prompt(
                tokenizer,
                SYSTEM_PROMPT,
                f"""Synthesize the following chronological partial notes.

Do not invent facts. Preserve exact financial figures, distinguish decisions
from proposals, and only assign action owners when explicitly supported.

Partial notes:
{combined}

Return the required seven Markdown sections.""",
            )

            next_level.append(
                generate_text(model, tokenizer, prompt, REDUCE_MAX_NEW_TOKENS)
            )

        current = next_level
        level += 1

    return current[0] if current else "No summary generated."


def generate_summary(transcript: str) -> str:
    print("\n[4/5] Loading local LLM...")
    tokenizer, model = load_local_llm()

    chunks = chunk_transcript_by_tokens(
        transcript,
        tokenizer,
        MAP_MAX_INPUT_TOKENS,
        MAP_OVERLAP_TOKENS,
    )

    print(
        f"[4/5] Created {len(chunks)} chronological token-aware chunk(s)."
    )

    chunk_summaries = [
        summarize_chunk(model, tokenizer, chunk, i, len(chunks))
        for i, chunk in enumerate(chunks, 1)
    ]

    print("[4/5] Starting recursive REDUCE...")
    final_summary = recursive_reduce(model, tokenizer, chunk_summaries)

    del model, tokenizer
    free_vram()

    return final_summary


In [10]:
from xml.sax.saxutils import escape as xml_escape
from docx import Document
from docx.shared import Pt
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    ListFlowable,
    ListItem,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle


def _strip_markdown_inline(text: str) -> str:
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"`(.*?)`", r"\1", text)
    return text.strip()


def export_docx(text: str, output_path: str):
    """Stage 5: Export the structured Markdown summary to Word."""
    doc = Document()
    doc.add_heading("Meeting Summary", 0)

    for line in text.splitlines():
        stripped = line.strip()

        if not stripped:
            continue

        if stripped.startswith("### "):
            doc.add_heading(_strip_markdown_inline(stripped[4:]), level=3)
        elif stripped.startswith("## "):
            doc.add_heading(_strip_markdown_inline(stripped[3:]), level=2)
        elif stripped.startswith("# "):
            doc.add_heading(_strip_markdown_inline(stripped[2:]), level=1)
        elif stripped.startswith("- "):
            doc.add_paragraph(
                _strip_markdown_inline(stripped[2:]),
                style="List Bullet",
            )
        else:
            doc.add_paragraph(_strip_markdown_inline(stripped))

    doc.save(output_path)
    print(f"[5/5] Saved DOCX: {output_path}")


def export_pdf(text: str, output_path: str):
    """Stage 5: Safer PDF export with escaped ReportLab text."""
    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=45,
        leftMargin=45,
        topMargin=45,
        bottomMargin=45,
    )

    styles = getSampleStyleSheet()
    styles.add(
        ParagraphStyle(
            name="MeetingHeading",
            parent=styles["Heading2"],
            spaceBefore=10,
            spaceAfter=6,
        )
    )

    story = [
        Paragraph("Meeting Summary", styles["Title"]),
        Spacer(1, 12),
    ]

    bullet_buffer = []

    def flush_bullets():
        nonlocal bullet_buffer
        if bullet_buffer:
            items = [
                ListItem(
                    Paragraph(xml_escape(_strip_markdown_inline(item)), styles["BodyText"])
                )
                for item in bullet_buffer
            ]
            story.append(ListFlowable(items, bulletType="bullet"))
            story.append(Spacer(1, 6))
            bullet_buffer = []

    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue

        if stripped.startswith("- "):
            bullet_buffer.append(stripped[2:])
            continue

        flush_bullets()

        if stripped.startswith("### "):
            story.append(
                Paragraph(
                    xml_escape(_strip_markdown_inline(stripped[4:])),
                    styles["Heading3"],
                )
            )
        elif stripped.startswith("## "):
            story.append(
                Paragraph(
                    xml_escape(_strip_markdown_inline(stripped[3:])),
                    styles["MeetingHeading"],
                )
            )
        elif stripped.startswith("# "):
            story.append(
                Paragraph(
                    xml_escape(_strip_markdown_inline(stripped[2:])),
                    styles["Heading1"],
                )
            )
        else:
            story.append(
                Paragraph(
                    xml_escape(_strip_markdown_inline(stripped)),
                    styles["BodyText"],
                )
            )
            story.append(Spacer(1, 4))

    flush_bullets()
    doc.build(story)
    print(f"[5/5] Saved PDF: {output_path}")


In [11]:
# ===========================================================================
# Pipeline Execution
# ===========================================================================

if not os.path.exists(AUDIO_FILE_PATH):
    raise FileNotFoundError(
        f"Audio file not found: {AUDIO_FILE_PATH}"
    )

print("=== Pipeline Initiated ===")
print("Input:", AUDIO_FILE_PATH)
print("Output:", OUTPUT_DOC_TYPE)

# Stage 1 — speaker diarization
diarization_data = diarize_audio(AUDIO_FILE_PATH)

# Stage 2 — transcription + chronological speaker alignment
raw_transcript = transcribe_and_align(
    AUDIO_FILE_PATH,
    diarization_data,
)

# Stage 3 — optional speaker naming
named_transcript = map_speakers_interactively(raw_transcript)

# Stage 4 — local LLM map/reduce
summary_markdown = generate_summary(named_transcript)

# Stage 5 — document export
base_filename = os.path.splitext(AUDIO_FILE_PATH)[0]

if OUTPUT_DOC_TYPE.lower() == "docx":
    output_path = f"{base_filename}_Summary.docx"
    export_docx(summary_markdown, output_path)
elif OUTPUT_DOC_TYPE.lower() == "pdf":
    output_path = f"{base_filename}_Summary.pdf"
    export_pdf(summary_markdown, output_path)
else:
    raise ValueError("OUTPUT_DOC_TYPE must be 'docx' or 'pdf'.")

print("\n=== Pipeline Complete ===")
print("Summary:", output_path)


FileNotFoundError: Audio file not found: meeting.mp3